# RAG: Generative Comparison & Reranker Fine-tuning

This notebook covers the advanced stages of the RAG pipeline:
1.  **Generative Comparison**: Llama 3.1 vs Qwen 2.5 (AWQ Quantized) answering questions based on retrieved context.
2.  **Fine-tuning**: Training a Cross-Encoder (Reranker) on the InsuranceQA dataset to improve retrieval metrics.

## Standalone Execution
**You do NOT need to run the previous benchmark notebook.** 
This notebook is self-contained. It will:
*   Read the raw data (`data/*.jsonl`) directly.
*   Build its own **in-memory retrieval index** (Qdrant) from scratch.
*   Download and run the LLMs automatically.

## Hardware Requirements
*   **GPU**: T4 (minimum) or A100 (recommended). 12GB+ VRAM required to load LLMs.

In [ ]:
# 1. Install Dependencies
# Note: autoawq is needed for the specific quantized models requested.
# If running locally on Windows, this might be tricky. On Colab, it works well.
import sys

print("Installing dependencies... (this may take a few minutes)")
!{sys.executable} -m pip install -q --upgrade transformers accelerate autoawq qdrant-client sentence-transformers datasets matplotlib seaborn pandas numpy

In [ ]:
# 2. Setup Data & Paths
import os
import json
import pandas as pd
from collections import namedtuple

# Find datasets
base_path = os.getcwd()
possible_paths = [
    os.path.join(base_path, "data"),
    os.path.join(base_path, "..", "data"),
    "/content/data" # Colab default
]

DATA_DIR = None
for path in possible_paths:
    if os.path.exists(path) and os.path.exists(os.path.join(path, "train.jsonl")):
        DATA_DIR = path
        print(f"Found data in: {DATA_DIR}")
        break

if not DATA_DIR:
    # Fallback for empty environment (download sample if needed, or raise error)
    print("Warning: Data not found. Please ensure 'data/train.jsonl' exists.")
    # On Colab you might need to upload data here
else:
    TRAIN_PATH = os.path.join(DATA_DIR, "train.jsonl")
    TEST_PATH = os.path.join(DATA_DIR, "test.jsonl")

In [ ]:
# 3. Retrieval Setup (Mini-Index)
# We need to retrieve contexts to feed the LLM. 
# We'll build a quick in-memory index for a subset of data to keep it fast.

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import numpy as np
import torch

# Configuration
EMBEDDING_MODEL_ID = "infgrad/stella-base-en-v2" 
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Embedding Model: {EMBEDDING_MODEL_ID} on {device}")
embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=device)

# Load a subset of Test data for the benchmark
def load_data_subset(path, limit=200):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= limit: break
            data.append(json.loads(line))
    return data

# We'll use the TEST set for questions, but we need a Knowledge Base.
# Typically RAG uses the TRAIN set or a separate corpus as the Knowledge Base.
# Here we'll index 1000 items from TRAIN to simulate the DB.
kb_data = load_data_subset(TRAIN_PATH, limit=1000)
test_queries = load_data_subset(TEST_PATH, limit=10) # 10 questions for GenAI Benchmark

# Indexing in Qdrant (In-Memory)
client = QdrantClient(":memory:")
COLLECTION_NAME = "insurance_benchmark"

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(size=embedder.get_sentence_embedding_dimension(), distance=models.Distance.COSINE)
)

print("Indexing Knowledge Base...")
documents = [item['output'] for item in kb_data]
vectors = embedder.encode(documents, convert_to_tensor=False, show_progress_bar=True)

client.upload_collection(
    collection_name=COLLECTION_NAME,
    vectors=vectors,
    payload=[{"text": doc} for doc in documents],
    ids=range(len(documents))
)

def retrieve_context(query, top_k=3):
    query_vector = embedder.encode(query, convert_to_tensor=False)
    hits = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k
    )
    return [hit.payload['text'] for hit in hits]

print("Retrieval System Ready.")

In [ ]:
# 4. LLM Benchmark (Llama 3.1 vs Qwen 2.5)
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc

def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

def generate_response(model, tokenizer, query, context_list):
    context_str = "\n\n".join(context_list)
    
    # Simple RAG Prompt
    # We use apply_chat_template for model-specific formatting
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question using ONLY the context provided below. If the context doesn't contain the answer, say 'I don't know'."},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=256, 
            temperature=0.1, # Low temperature for factual RAG
            do_sample=True
        )
        
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

def run_model_benchmark(model_id, dataset):
    print(f"\nLoading Model: {model_id}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        # Load in 4-bit using AutoAWQ (implied by model format) or bitsandbytes fallback
        model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
    except Exception as e:
        print(f"Failed to load {model_id}: {e}")
        return []

    results = []
    print("Generating answers...")
    for item in dataset:
        query = item['input']
        top_docs = retrieve_context(query)
        
        # English Answer
        ans = generate_response(model, tokenizer, query, top_docs)
        
        results.append({
            "Question": query,
            f"{model_id.split('/')[1]}_Answer": ans
        })
    
    # Cleanup
    del model
    del tokenizer
    clean_memory()
    print("Model unloaded.")
    return results

# Models to compare
# Note: You need access to Meta-Llama models on HF or use a mirror/quant.
# Ensure you are logged in via huggingface-cli login if using official Meta repos.
models_list = [
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    "Qwen/Qwen2.5-7B-Instruct-AWQ"
]

all_results = {}
for m in models_list:
    res = run_model_benchmark(m, test_queries)
    # Organization for the final table
    for r in res:
        q = r["Question"]
        ans_key = list(r.keys())[1]
        if q not in all_results: all_results[q] = {}
        all_results[q][ans_key] = r[ans_key]

# Display
df_gen = pd.DataFrame.from_dict(all_results, orient='index')
print(df_gen.head())
# df_gen.to_csv("llm_comparison_results.csv")

In [ ]:
# 5. Reranker Fine-tuning: Data Preparation
from sentence_transformers import InputExample
import random

def prepare_training_data(dataset_path, num_samples=2000):
    """
    Creates triplets: (Question, Positive, Negative)
    Negative is randomly sampled from other answers.
    """
    data = []
    with open(dataset_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    # Parse all
    parsed = [json.loads(line) for line in lines]
    
    # Create samples
    # Format for CrossEncoder: InputExample(texts=[q, doc], label=1.0) for Positive
    #                          InputExample(texts=[q, doc], label=0.0) for Negative
    train_samples = []
    
    all_answers = [p['output'] for p in parsed]
    
    for item in parsed[:num_samples]:
        question = item['input']
        pos_ans = item['output']
        
        # 1. Positive Pair
        train_samples.append(InputExample(texts=[question, pos_ans], label=1.0))
        
        # 2. Negative Pair (Random from corpus)
        # Simple Logic: unlikely to match
        neg_ans = random.choice(all_answers)
        while neg_ans == pos_ans:
            neg_ans = random.choice(all_answers)
            
        train_samples.append(InputExample(texts=[question, neg_ans], label=0.0))
        
    return train_samples

print("Preparing Training Data...")
train_examples = prepare_training_data(TRAIN_PATH)
print(f"Created {len(train_examples)} training pairs (Pos+Neg) from {len(train_examples)//2} questions.")

In [ ]:
# 6. Fine-tuning & Evaluation
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder.evaluation import CESoftmaxAccuracyEvaluator
import math

# Use the list-wise evaluation method for speed and isolation
# Task: Given [Correct_Answer + 9 Random_Negatives], can the reranker put Correct_Answer at #1?
def evaluate_reranker_isolated(model, dataset, num_negatives=9):
    hits = 0
    mrr = 0
    total = len(dataset)
    
    all_docs = [d['output'] for d in kb_data] # Reuse the 1000 docs loaded earlier as negatives source
    
    print(f"Evaluating on {total} samples...")
    for item in dataset:
        query = item['input']
        correct_ans = item['output']
        
        # Prepare Candidates
        negatives = random.sample(all_docs, num_negatives)
        candidates = [correct_ans] + negatives
        # Shuffle handled implicitly by sorting later? No, we need to know which index was 'correct_ans'
        # Let's simple format pairs
        
        pairs = [[query, doc] for doc in candidates]
        
        # Predict scores
        scores = model.predict(pairs)
        
        # The correct answer is at index 0 (in our candidates list)
        # We need its rank among sorted scores
        
        # Zip scores with original indices
        scored_candidates = list(zip(range(len(candidates)), scores))
        # Sort by score descending
        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        
        # Find where index 0 went
        rank = -1
        for r, (orig_idx, score) in enumerate(scored_candidates):
            if orig_idx == 0:
                rank = r + 1
                break
        
        if rank == 1:
            hits += 1
        mrr += 1.0 / rank

    return {"Accuracy@1": hits/total, "MRR": mrr/total}

# 1. Initialize Base Model
MODEL_NAME = "BAAI/bge-reranker-base"
reranker = CrossEncoder(MODEL_NAME, num_labels=1) # 1 label = regression/score or binary? BGE is trained for relevance score.
# Actually BGE Reranker output is logits. Using num_labels=1 usually works for simple scoring.

# Evaluate Baseline
print(">>> Baseline Performance (BGE-Base)")
# Create a robust test set (e.g. 50 items)
eval_subset = load_data_subset(TEST_PATH, limit=50) # Small for speed
base_metrics = evaluate_reranker_isolated(reranker, eval_subset)
print(base_metrics)

# 2. Train
# Define DataLoader
batch_size = 8 # Small batch for T4
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# BGE uses BCE loss usually? Or simply fitting.
# We trained with 1.0/0.0 labels, so we are doing regression or binary class.
# Default loss for CrossEncoder with num_labels=1 is usually MSE? 
# Let's force it cleanly. BAAI/bge usually produces raw logits.
print("\n>>> Starting Fine-tuning...")
reranker.fit(
    train_dataloader=train_dataloader,
    epochs=1,
    warmup_steps=100,
    output_path="models/my-finetuned-reranker",
    show_progress_bar=True
)

# 3. Evaluate Finetuned
print("\n>>> Finetuned Performance")
finetuned_metrics = evaluate_reranker_isolated(reranker, eval_subset)
print(finetuned_metrics)

# Comparison
print("\n=== Result Summary ===")
print(f"Baseline:  {base_metrics}")
print(f"Finetuned: {finetuned_metrics}")